# Emissions and GDP: 2008–2011

Use `NEI_clean_recession.parquet` to analyze **electric utilities** (sector 1) and **all other sectors** (sector 0 minus sector 1; wildfires remain excluded).

Income means `real_gdp_millions_2017usd` (total real GDP, not GDP per capita). The regression includes an intercept and **no state or year fixed effects**. It uses state-year levels from both years, not changes or national totals. Standard errors are clustered by state, with finite-sample correction and t-based p-values using the number of state clusters minus one degrees of freedom.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display

DATA_PATH = Path("NEI_clean_recession.parquet")
GDP = "real_gdp_millions_2017usd"
TOTAL = "sector_0_emissions_tons"
UTILITIES = "sector_1_emissions_tons"
df = pd.read_parquet(DATA_PATH)
required = ["state", "year", "pollutant", TOTAL, UTILITIES, GDP]
assert set(required).issubset(df.columns)
assert df[required].notna().all().all(), "Unexpected missing data."
assert set(df["year"]) == {2008, 2011}, "Expected exactly 2008 and 2011."
assert not df.duplicated(["state", "year", "pollutant"]).any()
assert np.isfinite(df[[TOTAL, UTILITIES, GDP]]).all().all()
assert df[[TOTAL, UTILITIES]].ge(0).all().all()
assert df[GDP].gt(0).all()
assert df.groupby(["state", "year"])[GDP].nunique().eq(1).all()
expected_states = set("AL AK AZ AR CA CO CT DE DC FL GA HI ID IL IN IA KS KY LA ME MD MA MI MN MS MO MT NE NV NH NJ NM NY NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY".split())
assert df.groupby(["pollutant", "year"])["state"].apply(lambda s: set(s) == expected_states).all()

df["other_emissions_tons"] = df[TOTAL] - df[UTILITIES]
assert df["other_emissions_tons"].ge(0).all(), "Utilities exceed total emissions."
sector_columns = {"Electric utilities": UTILITIES, "All others": "other_emissions_tons"}
df["state"] = df["state"].astype(str)
df["log_income"] = np.log(df[GDP])
print(f"Loaded {len(df):,} pollutant-state-year rows, covering 50 states plus DC.")

## 1. National totals and percentage changes

Percentage change = 100 × (2011 total / 2008 total − 1). Emissions are in tons and real GDP is in millions of 2017 US dollars. Zero emissions are retained in these totals.

In [ ]:
national_emissions = df.groupby(["pollutant", "year"])[list(sector_columns.values())].sum()
# GDP is common to both sector analyses; do not sum repeated pollutant rows.
state_gdp = df[["state", "year", GDP]].drop_duplicates(["state", "year"])
national_gdp = state_gdp.groupby("year")[GDP].sum()

change_rows = []
for pollutant in sorted(df["pollutant"].unique()):
    for sector, outcome in sector_columns.items():
        totals = national_emissions.loc[pollutant, outcome]
        assert totals.loc[2008] > 0
        change_rows.append({"Pollutant / series": pollutant, "Sector": sector,
                            "Units": "Tons", "2008": totals.loc[2008], "2011": totals.loc[2011]})
change_rows.append({"Pollutant / series": "Real GDP", "Sector": "Whole economy",
                    "Units": "Million 2017 USD", "2008": national_gdp.loc[2008],
                    "2011": national_gdp.loc[2011]})
changes = pd.DataFrame(change_rows).set_index(["Pollutant / series", "Sector"])
changes["Change (%)"] = 100 * (changes["2011"] / changes["2008"] - 1)
display(changes.style.format({"2008": "{:,.2f}", "2011": "{:,.2f}", "Change (%)": "{:+.2f}%"}))

# Export a compact LaTeX table of cumulative percentage changes.
labels = {"NOX": r"NO$_x$", "PM25": r"PM$_{2.5}$", "SO2": r"SO$_2$", "VOC": "VOCs"}
lines = [
    r"\begin{table}[htbp]",
    r"\centering",
    r"\caption{Percentage changes from 2008 to 2011}",
    r"\label{tab:recession-changes}",
    r"\small",
    r"\begin{tabular}{lrr}",
    r"\toprule",
    r"Series & Electric utilities & All other sectors \\",
    r"\midrule",
]
for pollutant in sorted(df["pollutant"].unique()):
    values = [changes.loc[(pollutant, sector), "Change (%)"] for sector in sector_columns]
    lines.append(labels.get(pollutant, pollutant) + " & " +
                 " & ".join(f"{v:+.2f}" + r"\%" for v in values) + r" \\")
gdp_change = changes.loc[("Real GDP", "Whole economy"), "Change (%)"]
lines += [
    r"\midrule",
    r"Real GDP & \multicolumn{2}{c}{" + f"{gdp_change:+.2f}" + r"\% (whole economy)} \\",
    r"\bottomrule",
    r"\end{tabular}",
    r"\par\smallskip",
    r"\begin{minipage}{0.75\linewidth}\footnotesize",
    r"\textit{Notes:} Cumulative changes in national totals (50 states and DC). "
    r"All other sectors equal total emissions minus electric utilities; wildfires are excluded. "
    r"Real GDP is counted once per state-year.",
    r"\end{minipage}",
    r"\end{table}",
]
output_path = Path("recession_percentage_changes.tex")
output_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"Saved {output_path}")


## 2. Pooled log–log regressions by sector and pollutant

Natural logarithms require strictly positive emissions. Display and exclude zero-emissions rows from regressions only; do not add a constant to emissions. Samples can therefore differ between sectors. Retain other valid observations from the same state.

In [ ]:
models, records = {}, []
for sector, outcome in sector_columns.items():
    excluded = df.loc[df[outcome].le(0), ["state", "year", "pollutant", outcome]]
    print(f"{sector}: {len(excluded)} rows excluded from log regressions.")
    display(excluded.reset_index(drop=True))
    for pollutant, group in df.loc[df[outcome].gt(0)].groupby("pollutant", sort=True):
        group = group.assign(log_emissions=np.log(group[outcome]))
        model = smf.ols("log_emissions ~ log_income", data=group, missing="raise").fit(
            cov_type="cluster",
            cov_kwds={"groups": group["state"], "use_correction": True, "df_correction": True},
            use_t=True,
        )
        assert model.model.exog_names == ["Intercept", "log_income"]
        assert int(model.nobs) == len(group)
        assert np.linalg.matrix_rank(model.model.exog) == 2
        models[(pollutant, sector)] = model
        records.append({"pollutant": pollutant, "sector": sector,
                        "elasticity": model.params["log_income"],
                        "clustered_se": model.bse["log_income"],
                        "p_value": model.pvalues["log_income"],
                        "observations": int(model.nobs), "state_clusters": group["state"].nunique(),
                        "r_squared": model.rsquared})
results = pd.DataFrame(records).set_index(["pollutant", "sector"])
assert np.isfinite(results).all().all()

## 3. Estimated elasticities

Each pollutant is a separate panel and each sector a column. Standard errors clustered by state appear in parentheses. Stars use two-sided clustered t-test p-values: * p < 0.10, ** p < 0.05, *** p < 0.01. Exact p-values are also reported.

In [ ]:
def stars(p):
    return "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else ""

rows, index = [], []
for pollutant in sorted(df["pollutant"].unique()):
    panel = results.loc[pollutant].reindex(sector_columns)
    entries = {
        "Income elasticity": [f"{r.elasticity:.3f}{stars(r.p_value)}" for r in panel.itertuples()],
        "State-clustered SE": [f"({r.clustered_se:.3f})" for r in panel.itertuples()],
        "p-value": [f"{p:.4g}" for p in panel["p_value"]],
        "Observations": [str(int(n)) for n in panel["observations"]],
        "State clusters": [str(int(n)) for n in panel["state_clusters"]],
        "R²": [f"{r:.3f}" for r in panel["r_squared"]],
    }
    for statistic, values in entries.items():
        index.append((pollutant, statistic))
        rows.append(values)
elasticity_table = pd.DataFrame(rows, columns=list(sector_columns),
    index=pd.MultiIndex.from_tuples(index, names=["Pollutant", "Statistic"]))
display(elasticity_table)
print("Intercept included; no state or year fixed effects. * p<0.10, ** p<0.05, *** p<0.01.")

# Export the section 3 results as a LaTeX table (requires booktabs).
latex_labels = {"NOX": r"NO$_x$", "PM25": r"PM$_{2.5}$", "SO2": r"SO$_2$", "VOC": "VOCs"}
latex_lines = [
    r"\begin{table}[htbp]",
    r"\centering",
    r"\caption{Income elasticities of emissions, 2008 and 2011}",
    r"\label{tab:recession-elasticities}",
    r"\small",
    r"\begin{tabular}{lrr}",
    r"\toprule",
    r" & Electric utilities & All other sectors \\",
]
for pollutant in sorted(results.index.get_level_values("pollutant").unique()):
    latex_lines += [r"\midrule",
                    r"\multicolumn{3}{l}{\textbf{" + latex_labels.get(pollutant, pollutant) + r"}} \\"]
    for statistic, values in elasticity_table.loc[pollutant].iterrows():
        label = r"$R^2$" if statistic == "R²" else statistic
        latex_lines.append(label + " & " + " & ".join(values.astype(str)) + r" \\")
latex_lines += [
    r"\bottomrule",
    r"\end{tabular}",
    r"\par\smallskip",
    r"\begin{minipage}{0.85\linewidth}\footnotesize",
    r"\textit{Notes:} Separate unweighted pooled OLS regressions of log emissions on log real GDP "
    r"using state-year observations for 2008 and 2011. Intercept included; no state or year fixed effects. "
    r"Standard errors clustered by state (including DC), with finite-sample correction; "
    r"two-sided $t$-tests use the number of clusters minus one degrees of freedom. "
    r"Nonpositive emissions are excluded. All other sectors equal total emissions minus electric utilities; "
    r"wildfires are excluded. * $p<0.10$, ** $p<0.05$, *** $p<0.01$.",
    r"\end{minipage}",
    r"\end{table}",
]
elasticity_output = Path("recession_elasticities.tex")
elasticity_output.write_text("\n".join(latex_lines) + "\n", encoding="utf-8")
print(f"Saved {elasticity_output}")
